# Fit LG — Twitch community graphs (`twitch`)

Batch fit and model comparison for **all networks** in this collection.

Six MUSAE Twitch community graphs (`data/twitch/graphs_processed/*_graph.edges`), sorted by |V|.

**Per network**, the fit loop logs four steps and saves a 4-panel report (`runs/twitch/{graph}/fit_report.png`):

1. **Load** — graph size and structural attributes  
2. **AIC** — pick `d̂` over candidates `[0,1,2,3]`  
3. **σ̂** — Layer-2 offset logit at `d̂`  
4. **Compare** — LG vs ER / WS / BA (`GraphModelComparator`, spectral GIC)

Reports are also displayed inline (only 6 networks). Aggregate outputs: `runs/twitch/`.

## 1. Discover networks

In [3]:
import os
import sys
import warnings
from pathlib import Path

for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")
warnings.filterwarnings("ignore", category=DeprecationWarning)

_REFACTOR = Path.cwd()
if str(_REFACTOR) not in sys.path:
    sys.path.insert(0, str(_REFACTOR))

from platform_fit_utils import PlatformConfig, discover_graph_files, print_discovery

# ---- Batch settings (edit here) --------------------------------------------
MIN_NODES = 0
MAX_NODES = 10_000      # skip graphs with n > MAX_NODES; set None for no cap
USE_CACHE = True     # reload finished networks from runs/{platform}/{graph}/

cfg = PlatformConfig(
    platform="twitch",
    glob_pattern="twitch/graphs_processed/*_graph.edges",
    min_nodes=MIN_NODES,
    max_nodes=MAX_NODES,
    use_cache=USE_CACHE,
    display_plots=True,
)

graph_files = discover_graph_files(cfg)
print_discovery(cfg, graph_files)

PLATFORM=twitch  RUN_DIR=/Users/maruanottoni/home/master/research/all_logit/logit-graph/notebooks/refactors/runs/twitch
Found 6 networks (MIN_NODES=0, MAX_NODES=10000, USE_CACHE=True)
                PTBR_graph.edges  n= 1912  |E|=  31299  [cached]
                  RU_graph.edges  n= 4385  |E|=  37304
                  ES_graph.edges  n= 4648  |E|=  59382
                  FR_graph.edges  n= 6549  |E|= 112666
                ENGB_graph.edges  n= 7126  |E|=  35324
                  DE_graph.edges  n= 9498  |E|= 153138


## 2. Fit all networks

In [ ]:
from platform_fit_utils import fit_all_networks

comparators, summary_all, fit_meta, failures = fit_all_networks(graph_files, cfg)
RUN_DIR = cfg.run_dir

19:37:33  === twitch batch fit  (6 networks, MAX_NODES=10000, cache=True) ===
19:37:33  [1/6] PTBR_graph  CACHED  (n=1912, best=ER, GIC=3.107)
19:37:33  [2/6] RU_graph


## 3. Aggregate comparison

In [ ]:
from platform_fit_utils import summarize_aggregates

gic_pivot, rank_pivot, mean_rank = summarize_aggregates(summary_all, RUN_DIR)

## 4. Summary plots

In [ ]:
from platform_fit_utils import plot_aggregate_summary

plot_aggregate_summary(fit_meta, mean_rank, RUN_DIR, cfg.platform, display=True)